In [1]:
import polars as pl
from extract import endpoints, get
from transform import transform_sports, transform_leagues, transform_seasons, SportSchema, LeagueSchema, SportCollection, SeasonSchema
from transform import DivisionSchema, DivisionSeasonsSchema
from transform import transform_divisions
from pathlib import Path

In [2]:
data = Path('data')
tables = endpoints.keys()
paths = [data/(table+'.parquet') for table in tables]

In [3]:
for table, path in zip(tables, paths):
    if not path.exists():
        get(table).collect().write_parquet(path)

In [4]:
sports = pl.scan_parquet(data/'sports.parquet')
leagues = pl.scan_parquet(data/'leagues.parquet')
divisions = pl.scan_parquet(data/'divisions.parquet')
seasons = pl.scan_parquet(data/'seasons.parquet', try_parse_hive_dates=True)

In [5]:
sports = transform_sports(sports)
sports = SportSchema.validate(sports, cast=True).lazy()

In [6]:
leagues = transform_leagues(leagues)
leagues = LeagueSchema.validate(leagues, cast=True).lazy()

In [7]:
SC, _ = SportCollection.filter(
    {
        'sports': sports,
        'leagues': leagues,
    }
)

In [8]:
lf = pl.scan_parquet('data/seasons.parquet')
lf = transform_seasons(lf)
seasons = SeasonSchema.validate(lf, cast=True).lazy()

In [14]:
div, div_seasons = transform_divisions(divisions, leagues)

In [15]:
divisions = DivisionSchema.validate(div, cast=True).lazy()
division_seasons = DivisionSeasonsSchema.validate(div_seasons, cast=True).lazy()